# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 8: Förderung (aktive Arbeitsmarktpolitik)

Letzter Punkt aus dem Priorisierungsvorschlag in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md): arbeitsmarktpolitische Maßnahmen, mit denen das Jobcenter Menschen aktiv in Arbeit oder Ausbildung unterstützt (z. B. Weiterbildung, Vermittlungsbudget, Eingliederungszuschuss).

**Zwei neue Tabellenblätter, direkt aus den Excel-Rohdaten gelesen** (wie schon Kapitel 6 und 7):
- **„4.1 Förderung“**: Eintritte in Maßnahmen (Zugang) und Bestand an Teilnehmenden, inkl. gleitender 12-Monatswerte.
- **„4.2 Förderung Strukturen“**: Struktur des Bestands nach Geschlecht, Alter und LZA-Status vor Eintritt, sowie die fertige Kennzahl **Aktivierungsquote (AQ1/AQ2a)**.

⚠️ **Wartezeit:** wie bei den Kapiteln 6 und 7 nur ein Datenblock mit 3 Monaten Wartezeit → Zeitreihe für **Oktober 2024 – Juni 2026**.

⚠️ **Anonymisierung:** Die Förderstatistik unterdrückt kleine Fallzahlen (< 3) als „*“ — betrifft vor allem die feineren Instrumenten-Unterkategorien. Dieses Notebook beschränkt sich daher bewusst auf die **Summenzeilen**, die in allen 21 Dateien durchgehend als Zahl vorliegen (keine Unterdrückung), statt auf einzelne kleine Maßnahmen-Kategorien. Das ist auch deshalb robust, weil die Bundesagentur ab dem Bericht August 2026 (Datenmonat Mai 2026) eine Unterkategorie umbenannt hat („Eingliederung von Langzeit**arbeitslosen**“ → „Eingliederung von Langzeit**leistungsbeziehenden**“) — die hier genutzten Summenzeilen sind davon nicht betroffen.

Außerdem gilt: Die Förderstatistik zählt **Förderfälle, nicht Personen** — wer mehrere Leistungen gleichzeitig erhält, wird mehrfach gezählt.

In [1]:
import re
from pathlib import Path

import openpyxl
import pandas as pd
import plotly.graph_objects as go

RAW_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"

MONATE = {
    "Januar": 1, "Februar": 2, "März": 3, "April": 4, "Mai": 5, "Juni": 6,
    "Juli": 7, "August": 8, "September": 9, "Oktober": 10, "November": 11, "Dezember": 12,
}


def parse_monat(text):
    monat_name, jahr = re.match(r"(\w+)\s+(\d{4})", text).groups()
    return pd.Period(f"{jahr}-{MONATE[monat_name]:02d}", freq="M")


def parse_foerderung(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws41 = wb["4.1 Förderung"]
    ws42 = wb["4.2 Förderung Strukturen"]
    monat = parse_monat(ws41["A6"].value)
    zeile = {
        "period": monat,
        "Eintritte (Summe)": ws41["B39"].value,
        "Eintritte 12M-Summe": ws41["F39"].value,
        "Eintritte ohne Einmalleistungen": ws41["B40"].value,
        "Bestand (Summe)": ws41["B58"].value,
        "Bestand 12M-Durchschnitt": ws41["F58"].value,
        "Bestand Männer": ws42["C39"].value,
        "Bestand Frauen": ws42["D39"].value,
        "Bestand 15-25": ws42["E39"].value,
        "Bestand 25-55": ws42["F39"].value,
        "Bestand 55+": ws42["G39"].value,
        "Bestand vor Eintritt LZA": ws42["H39"].value,
        "AQ1 (%)": ws42["B56"].value,
        "AQ1 12M-Durchschnitt (%)": ws42["B57"].value,
        "AQ2a (%)": ws42["B58"].value,
        "AQ2a 12M-Durchschnitt (%)": ws42["B59"].value,
    }
    wb.close()
    return zeile


foerderung = pd.DataFrame(
    [parse_foerderung(p) for p in sorted(RAW_DIR.glob("*.xlsx"))]
).set_index("period").sort_index()
foerderung.tail()

ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"


def export_abbildung(fig, dateiname, width, height, **layout):
    """Speichert eine Kopie der Grafik im weissen Layout der Projektdokumentation
    (Docs/Abbildungen, doppelte Aufloesung); die Anzeige im Notebook bleibt unveraendert.
    Optionale `layout`-Angaben (z. B. kuerzerer Titel) gelten nur fuer die exportierte Datei."""
    go.Figure(fig).update_layout(template="plotly_white", **layout).write_image(
        ABBILDUNGEN_DIR / dateiname, scale=2, width=width, height=height
    )

C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


## 1. Datenüberblick und Plausibilitätsprüfung

Der Bestand an Teilnehmenden wird in beiden Blättern unabhängig ausgewiesen — einmal als reine Summenzeile in „4.1“, einmal als Ausgangswert der Strukturaufschlüsselung in „4.2“. Beide sollten exakt übereinstimmen.

In [2]:
print(f"Anzahl Monate: {len(foerderung)}")
print(f"Zeitraum: {foerderung.index.min()} bis {foerderung.index.max()}")
print(f"Duplikate: {foerderung.index.duplicated().sum()}")

gender_check = foerderung["Bestand Männer"] + foerderung["Bestand Frauen"] - foerderung["Bestand (Summe)"]
alter_check = foerderung["Bestand 15-25"] + foerderung["Bestand 25-55"] + foerderung["Bestand 55+"] - foerderung["Bestand (Summe)"]
print(f"Geschlecht-Summen-Check (Männer+Frauen vs. Bestand): min={gender_check.min()}, max={gender_check.max()}")
print(f"Alter-Summen-Check (15-25+25-55+55+ vs. Bestand): min={alter_check.min()}, max={alter_check.max()}")

Anzahl Monate: 21
Zeitraum: 2024-10 bis 2026-06
Duplikate: 0
Geschlecht-Summen-Check (Männer+Frauen vs. Bestand): min=0, max=0
Alter-Summen-Check (15-25+25-55+55+ vs. Bestand): min=-11, max=0


Alle 21 Dateien liefern 21 lückenlose, eindeutige Monate (Oktober 2024 – Juni 2026, keine Duplikate). Der Bestand aus beiden Blättern stimmt in jedem Monat exakt überein (4.1 „Summe der Instrumente“ = 4.2 „Insgesamt“) — ein gutes Konsistenz-Signal. Auch die Aufteilung nach Geschlecht summiert sich exakt zum Bestand. Die Aufteilung nach Alter weicht dagegen in einzelnen Monaten um bis zu 11 Personen nach unten ab — vermutlich eine kleine Gruppe mit unbekanntem Alter, die in der Alterssystematik nicht separat ausgewiesen wird.

## 2. Eintritte und Bestand an Fördermaßnahmen im Zeitverlauf

In [3]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=foerderung.index.to_timestamp(), y=foerderung["Eintritte (Summe)"], name="Eintritte (im Monat)",
    marker_color="#f59e0b", yaxis="y1",
))
fig.add_trace(go.Scatter(
    x=foerderung.index.to_timestamp(), y=foerderung["Bestand (Summe)"], name="Bestand (im Monat)",
    mode="lines+markers", line=dict(color="#2563eb"), marker=dict(size=6), yaxis="y2",
))
fig.update_layout(
    title="Eintritte und Bestand an Teilnehmenden in Fördermaßnahmen (SGB II)",
    yaxis=dict(title="Eintritte im Monat"),
    yaxis2=dict(title="Bestand im Monat", overlaying="y", side="right"),
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=460,
)
export_abbildung(fig, "abb22_eintritte_bestand.png", 1000, 480)
fig

Die monatlichen **Eintritte** in Fördermaßnahmen schwanken deutlich (49 bis 119, Ø 86,7 pro Monat) und sind insgesamt rückläufig (119 im Oktober 2024, 73 im Juni 2026). Der **Bestand** an Teilnehmenden ist dagegen zunächst spürbar gestiegen — von 229 (Oktober 2024) auf einen Höchststand von 313 (April 2025) —, bis März 2026 wieder auf 250 gesunken und seither erneut deutlich gestiegen auf 291 (Juni 2026). Der gleitende 12-Monatsdurchschnitt des Bestands (203,0 → 263,8, Höchstwert 283,0 im September 2025) zeigt den zugrunde liegenden Trend klarer als die volatilen Monatswerte: eine deutliche Ausweitung der Förderaktivität gegenüber der Ausgangslage Ende 2024.

## 3. Aktivierungsquote im Zeitverlauf

Die **Aktivierungsquote** ist eine von der Bundesagentur direkt mitgelieferte, bestandsorientierte Kennzahl: Anteil der aktivierten (in Maßnahmen befindlichen) Personen an
- **AQ1**: allen Teilnehmenden plus Arbeitslosen,
- **AQ2a**: allen erwerbsfähigen Leistungsberechtigten (ELB).

Beide Varianten werden hier zusätzlich als gleitender 12-Monatsdurchschnitt gezeigt, um saisonale Schwankungen zu glätten.

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=foerderung.index.to_timestamp(), y=foerderung["AQ1 (%)"], name="AQ1", mode="lines+markers", line=dict(color="#2563eb"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=foerderung.index.to_timestamp(), y=foerderung["AQ1 12M-Durchschnitt (%)"], name="AQ1 (12M-Ø)", mode="lines", line=dict(color="#2563eb", dash="dot")))
fig.add_trace(go.Scatter(x=foerderung.index.to_timestamp(), y=foerderung["AQ2a (%)"], name="AQ2a", mode="lines+markers", line=dict(color="#7c3aed"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=foerderung.index.to_timestamp(), y=foerderung["AQ2a 12M-Durchschnitt (%)"], name="AQ2a (12M-Ø)", mode="lines", line=dict(color="#7c3aed", dash="dot")))
fig.update_layout(
    title="Aktivierungsquote AQ1 und AQ2a im Zeitverlauf (SGB II)",
    yaxis_title="Prozent", yaxis_ticksuffix="%",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=460,
)
export_abbildung(fig, "abb23_aktivierungsquote.png", 1000, 460)
fig

Beide Aktivierungsquoten sind bis Frühjahr 2026 spürbar **gesunken** — AQ1 von 13,3 % (Oktober 2024) auf den Tiefstwert 9,9 % (März 2026) — und haben sich seither wieder **erholt**: Im Juni 2026 liegt AQ1 bei 11,8 % (Ø 12,2 %, Spanne 9,9–15,0 %), AQ2a mit 6,6 % sogar über dem Ausgangswert von 6,0 % (Ø 6,3 %, Spanne 5,5–7,4 %). Die geglätteten 12-Monatsdurchschnitte zeigen das differenziertere Bild: Der AQ1-Durchschnitt sinkt (12,3 % → 10,9 %), während der AQ2a-Durchschnitt steigt (5,3 % → 6,1 %). Der jüngste Wiederanstieg spiegelt sich in den 12-Monatswerten noch kaum wider. Insgesamt bleibt die Aktivierung ein eher kleiner Teil der SGB-II-Grundgesamtheit: Nur rund jede zehnte Person aus dem AQ1-Bezug (Arbeitslose + Teilnehmende) und gut jede zwanzigste ELB nimmt an einer Fördermaßnahme teil.

## 4. Struktur des Bestands: Geschlecht und Vorgeschichte

Anteil der Teilnehmenden, die vor Eintritt in die Maßnahme bereits langzeitarbeitslos waren — ein möglicher Anknüpfungspunkt an den in [`03_Statistik.ipynb`](03_Statistik.ipynb) beschriebenen Anstieg der Langzeitarbeitslosigkeit.

In [5]:
anteil_frauen = (foerderung["Bestand Frauen"] / foerderung["Bestand (Summe)"] * 100).round(1)
anteil_lza = (foerderung["Bestand vor Eintritt LZA"] / foerderung["Bestand (Summe)"] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=foerderung.index.to_timestamp(), y=anteil_frauen, name="Anteil Frauen am Bestand (%)", mode="lines+markers", line=dict(color="#db2777"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=foerderung.index.to_timestamp(), y=anteil_lza, name="Anteil vor Eintritt langzeitarbeitslos (%)", mode="lines+markers", line=dict(color="#dc2626"), marker=dict(size=5)))
fig.update_layout(
    title="Bestand an Fördermaßnahmen-Teilnehmenden: Anteile im Zeitverlauf",
    yaxis_title="Prozent", yaxis_ticksuffix="%",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=440,
)
export_abbildung(fig, "abb24_foerderung_anteile.png", 1000, 440)
fig

Der **Anteil Frauen** am Bestand liegt über den Zeitraum überwiegend bei rund 44 % (44,5 % → 44,3 %) — Männer sind unter den Fördermaßnahmen-Teilnehmenden also durchgehend leicht in der Mehrheit. Deutlich auffälliger: Der **Anteil der zuvor Langzeitarbeitslosen** unter den Teilnehmenden steigt spürbar von 16,2 % auf 25,1 % (Höchstwert 26,3 % im Mai 2026) — ein Befund, der zur in [`03_Statistik.ipynb`](03_Statistik.ipynb) beschriebenen Verschärfung der Langzeitarbeitslosigkeit passt: Das Jobcenter aktiviert einen wachsenden Anteil seiner Fördermaßnahmen gezielt für Personen, die bereits langzeitarbeitslos waren.

## 5. Export der aufbereiteten Daten

Grundlage für den Dashboard-Tab „🎯 Förderung“.

In [6]:
foerderung.to_csv(PROCESSED_DIR / "foerderung_sgb2.csv")
foerderung.tail()

,Eintritte (Summe),Eintritte 12M-Summe,Eintritte ohne Einmalleistungen,Bestand (Summe),Bestand 12M-Durchschnitt,Bestand Männer,Bestand Frauen,Bestand 15-25,Bestand 25-55,Bestand 55+,Bestand vor Eintritt LZA,AQ1 (%),AQ1 12M-Durchschnitt (%),AQ2a (%),AQ2a 12M-Durchschnitt (%)
period,,,,,,,,,,,,,,,
2026-02,69,992,52,246,272.500000,135,111,65,156,20,56,10.139860,11.868046,5.679513,6.352503
2026-03,91,971,64,250,268.666667,141,109,61,157,21,59,9.924110,11.506276,5.512489,6.248802
2026-04,94,976,63,263,264.500000,148,115,67,165,22,67,10.520297,11.175300,5.945166,6.136473
2026-05,84,976,60,281,263.916667,159,122,67,179,26,74,11.415525,11.027884,6.395349,6.117188
2026-06,73,961,51,291,263.750000,162,129,73,181,29,73,11.764706,10.907771,6.619385,6.092401


## 6. Zusammenfassung

- **Eintritte:** volatil, insgesamt rückläufig (Ø 86,7 pro Monat, 119 → 73).
- **Bestand:** deutlich gestiegen bis Frühjahr 2025 (Höchststand 313 im April 2025), bis März 2026 wieder gesunken auf 250, seither erneut gestiegen auf 291 (Juni 2026); der geglättete 12-Monatsdurchschnitt zeigt den Anstieg gegenüber 2024 klar (203,0 → 263,8).
- **Aktivierungsquoten:** bis März 2026 rückläufig (AQ1: 13,3 % → 9,9 %), seither wieder steigend (AQ1 11,8 %, AQ2a 6,6 % im Juni 2026) — nur ein kleiner Teil der SGB-II-Grundgesamtheit nimmt an Fördermaßnahmen teil.
- **Struktur:** Geschlechterverteilung überwiegend stabil (~44 % Frauen), aber der Anteil zuvor langzeitarbeitsloser Teilnehmender steigt deutlich (16,2 % → 25,1 %) — passend zur insgesamt zunehmenden Langzeitarbeitslosigkeit im Datensatz.
- **Datenbesonderheit:** Zeitreihe Oktober 2024 – Juni 2026 (3 Monate Wartezeit), analog zu Kapitel 6 und 7; kleine Fallzahlen (< 3) werden von der Bundesagentur anonymisiert, weshalb sich dieses Notebook auf robuste Summenzeilen beschränkt.

Insgesamt zeigt sich: Das Jobcenter Regensburg hat seine Förderaktivität gegenüber Ende 2024 spürbar ausgeweitet und richtet sie zunehmend gezielt an Personen mit Langzeitarbeitslosigkeits-Vorgeschichte. Nach einem Rückgang der Aktivierungsquoten bis Frühjahr 2026 zieht die Förderaktivität seit April 2026 wieder spürbar an.

Damit ist die im Notebook [`01_Exploration.ipynb`](01_Exploration.ipynb) begonnene und in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md) priorisierte Themenliste vollständig abgearbeitet. Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py), Tab „🎯 Förderung“. Weitere offene Themen (z. B. Bezugsdauer/Langzeitleistungsbezug, gemeldete Arbeitsstellen) stehen weiterhin in `ArbeitslosenAnalyse.md`.